# Word LLM Translation Workflow — Primary Translation Analysis Stage

- **Workflow stage:** analysis of the primary translation checkpoint
- **Input checkpoint:** primary translation or primary-retry checkpoint from the previous stage
- **Source document:** loaded from checkpoint metadata
- **Source language:** loaded from checkpoint metadata
- **Target language:** loaded from checkpoint metadata
- **Primary translator model:** loaded from checkpoint metadata
- **Purpose of this notebook:** inspect the primary translation state before moving on to evaluation

## Purpose of this notebook
This notebook performs a lightweight QA and inspection pass on the primary translation output before evaluation begins. It loads a saved checkpoint, checks the translated elements for obvious issues, reviews basic formatting and language-detection signals, and can generate a Word document using the primary translation output.

## What this notebook checks
This notebook currently includes checks such as:

- loading and inspecting checkpoint metadata and element structure
- identifying unchanged source/translation pairs
- flagging likely English leakage in the translated text using heuristic and `langid`-based methods
- checking for missing or blank `primary_translation` values
- inspecting suspicious source/translation length ratios
- exporting a Word document built from the `primary_translation` field

## Notes
- This notebook is still operating on the `primary_translation` stage, not the evaluated or finalized output.
- The Word export generated here is useful for quick review, but it is not the final delivery artifact.
- Evaluation and fallback logic are handled in later notebooks.

### List contents of */checkpoints* subdirectory

In [1]:
# show json files in the /checkpoints subdirectory
import importlib
import workflow_helpers
workflow_helpers = importlib.reload(workflow_helpers)

checkpoints_dir = workflow_helpers.get_checkpoints_dir()
json_files = workflow_helpers.list_json_files(checkpoints_dir)

Found JSON files:
- elements_batched_20260411_1519.json
- primary_retry_completed_1_20260411_1541.json
- primary_translation_completed_20260411_1527.json


### Load json state file
Enter the json filename (either has primary_translation_completed or primary_retry_completed in the filename) you wish to analyze in the json_file variable in the cell below

In [2]:
# Load checkpoint state (metadata + elements)

import os
from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

# Define the checkpoint file
checkpoint_dir = "checkpoints"
json_file = "primary_retry_completed_1_20260411_1541.json"

# Full path
checkpoint_path = os.path.join(checkpoint_dir, json_file)

# Load metadata and elements
metadata, elements = workflow_helpers.load_elements_checkpoint(checkpoint_path)

print("Loaded checkpoint:", checkpoint_path)
print("Stage:", metadata.get("stage"))
print("Source language:", metadata.get("source_language"))
print("Target language:", metadata.get("target_language"))
print("Primary model:", metadata.get("primary_model_name"))
print(f"Loaded checkpoint with {len(elements)} elements.")
print("Example entry:")
elements[0]

Loaded checkpoint: checkpoints\primary_retry_completed_1_20260411_1541.json
Stage: primary_retry_completed
Source language: English
Target language: Traditional Chinese
Primary model: gemini-3.1-pro-preview
Loaded checkpoint with 29 elements.
Example entry:


{'element_number': 1,
 'element_type': 'paragraph',
 'word_style': 'h1',
 'text': 'Introduction',
 'element_id': '8633e724fc99',
 'tokens': 1,
 'batch_number': 1,
 'primary_translation': '簡介',
 'primary_translation_model': 'gemini-3.1-pro-preview',
 'primary_error': None,
 'evaluator_ran': None,
 'evaluator_passed': None,
 'evaluator_feedback': None,
 'evaluator_error': None,
 'fallback_translation': None,
 'fallback_translation_model': None,
 'fallback_error': None,
 'final': None,
 'final_model': None}

## Primary translation analysis

This section reviews the `primary_translation` output for obvious issues before evaluation, including unchanged elements, likely English leakage, missing translations, and other simple quality signals.

In [3]:
unchanged = [
    el for el in elements
    if isinstance(el.get("text"), str)
    and isinstance(el.get("primary_translation"), str)
    and el["text"].strip() == el["primary_translation"].strip()
]

len(unchanged)

0

In [4]:
for el in unchanged[:10]:
    print(f"ID: {el['element_id']}")
    print(f"EN: {el['text']}")
    print(f"RU: {el['primary_translation']}")
    print("-" * 60)

In [5]:
# create df for the unchanged entries
import pandas as pd

df_unchanged = pd.DataFrame(unchanged)
df_unchanged.head(10)


""


### Check for likely English leakage in `primary_translation`
This section looks for translated elements that may still contain substantial English text and therefore may need closer review.

#### Heuristic method: ASCII-heavy text detection
This quick check flags elements whose translated text contains a high proportion of ASCII characters, which can be a useful rough signal that English may still be present.

In [6]:
import pandas as pd

def is_likely_english(text, min_alpha=10, ascii_ratio_threshold=0.85):
    """
    Heuristic, target-language-agnostic:
    - Ignore very short strings with < min_alpha alphabetic chars.
    - Compute fraction of alphabetic chars that are ASCII a–z.
    - If that fraction >= ascii_ratio_threshold, flag as likely English.
    """
    if not isinstance(text, str):
        return False

    letters = [ch for ch in text if ch.isalpha()]
    if len(letters) < min_alpha:
        return False  # too short / code-like; we don't care

    ascii_letters = sum("a" <= ch.lower() <= "z" for ch in letters)
    ratio = ascii_letters / len(letters)

    return ratio >= ascii_ratio_threshold

# Build list of elements where primary_translation is probably English
suspect_english = [
    el for el in elements
    if is_likely_english(el.get("primary_translation", ""))
]

len(suspect_english)

0

In [7]:
## uncomment if suspect_english is >0
# for el in suspect_english[:20]:
#     print(f"ID:  {el['element_id']}")
#     print(f"EN:  {el['text']}")
#     print(f"TR:  {el['primary_translation']}")
#     print("-" * 80)

In [8]:
## uncomment to create a df of suspect_english
# df_suspect_english = pd.DataFrame(suspect_english)
# df_suspect_english.head(20)

#### Langid method
This method applies language identification to each translated element to estimate whether the text is more likely to be English than the intended target language.

In [9]:
import langid
import pandas as pd

# Optional: let langid use full language set (future-proof)
# langid.set_languages(None)

suspect_langid_en = []

for el in elements:
    txt = el.get("primary_translation", "")
    if not isinstance(txt, str) or not txt.strip():
        continue

    letters = [ch for ch in txt if ch.isalpha()]
    if len(letters) < 10:
        continue  # skip ultra-short things

    lang, score = langid.classify(txt)
    if lang == "en":
        el_copy = el.copy()
        el_copy["detected_lang"] = lang
        el_copy["lang_score"] = score
        suspect_langid_en.append(el_copy)

len(suspect_langid_en)

0

In [10]:
## uncomment to create df of suspect_langid
# df_suspect_langid_en = pd.DataFrame(suspect_langid_en)
# df_suspect_langid_en.head(20)

In [11]:
## uncomment if more than 0 results 
## intersect both methods

# ids_heuristic = {el["element_id"] for el in suspect_english}
# high_conf_suspect = [
#     el for el in suspect_langid_en if el["element_id"] in ids_heuristic
# ]

# len(high_conf_suspect)

### Formatting and style analytics
This section applies a series of lightweight checks to the `primary_translation` field to surface possible formatting or structure problems before evaluation. These checks look for issues such as blank entries, unusual truncation or expansion, newline anomalies, emphasis-marker mismatches, URL handling problems, and possible list-formatting inconsistencies, and then conclude with a short summary of findings.

In [12]:
# convert elements to df
import pandas as pd
df = pd.DataFrame(elements)

In [13]:
# Any missing or blank primary_translation?
mask_empty = df['primary_translation'].isna() | (df['primary_translation'].str.strip() == "")
df_empty = df[mask_empty]
len(df_empty), df_empty.head(10)

(0,
 Empty DataFrame
 Columns: [element_number, element_type, word_style, text, element_id, tokens, batch_number, primary_translation, primary_translation_model, primary_error, evaluator_ran, evaluator_passed, evaluator_feedback, evaluator_error, fallback_translation, fallback_translation_model, fallback_error, final, final_model]
 Index: [])

In [14]:
# Any unusual trunctions or expansions?
df['len_src'] = df['text'].str.len()
df['len_tr'] = df['primary_translation'].str.len()

# Avoid division by zero
df['len_ratio'] = df['len_tr'] / df['len_src'].replace({0: pd.NA})

# Suspiciously short translations for non-trivial source
suspect_short = df[(df['len_src'] >= 40) & (df['len_ratio'] < 0.4)]

# Suspiciously long translations (might be okay, but worth a look)
suspect_long  = df[(df['len_src'] >= 40) & (df['len_ratio'] > 3.0)]
len(suspect_short), len(suspect_long)

(21, 0)

In [15]:
# uncomment if results are greater than 0
suspect_short[['element_number', 'text', 'primary_translation']].head(20)
suspect_long[['element_number', 'text', 'primary_translation']].head(20)

,element_number,text,primary_translation


In [16]:
# inspect new line characters
df['nl_src'] = df['text'].str.count('\n')
df['nl_tr']  = df['primary_translation'].str.count('\n')

suspect_newlines = df[(df['nl_src'] != df['nl_tr']) & (df['nl_src'] > 0)]
len(suspect_newlines)

0

In [17]:
# suspect_newlines[['element_number', 'text', 'primary_translation']].head(20)

In [18]:
# Inspect * marker counts
import re
df['stars_src'] = df['text'].str.count(r'\*')
df['stars_tr']  = df['primary_translation'].str.count(r'\*')

star_mismatch = df[df['stars_src'] != df['stars_tr']]
len(star_mismatch)

0

In [19]:
star_mismatch[['element_number', 'text', 'primary_translation']].head(20)

,element_number,text,primary_translation


In [20]:
# select an index number to inspect
elements[10]

{'element_number': 11,
 'element_type': 'paragraph',
 'word_style': 'quotation',
 'text': 'You will need a **Bible** and a **pencil** or **highlighter**.',
 'element_id': 'fb49158b9536',
 'tokens': 19,
 'batch_number': 4,
 'primary_translation': '你需要一本**聖經**和一支**鉛筆**或**螢光筆**。',
 'primary_translation_model': 'gemini-3.1-pro-preview',
 'primary_error': None,
 'evaluator_ran': None,
 'evaluator_passed': None,
 'evaluator_feedback': None,
 'evaluator_error': None,
 'fallback_translation': None,
 'fallback_translation_model': None,
 'fallback_error': None,
 'final': None,
 'final_model': None}

In [21]:
# inspect URLs
df['has_url_src'] = df['text'].str.contains(r'http[s]?://', regex=True, na=False)
df['has_url_tr']  = df['primary_translation'].str.contains(r'http[s]?://', regex=True, na=False)

url_lost = df[(df['has_url_src']) & (~df['has_url_tr'])]
len(url_lost)

0

In [22]:
# url_lost[['element_number', 'text', 'primary_translation']].head(20)

In [23]:
# ordered list inspection (may miss some ol items)

# Very simple: lines starting with digit + '.'
df['starts_with_num_src'] = df['text'].str.match(r'^\s*\d+\.', na=False)
df['starts_with_num_tr']  = df['primary_translation'].str.match(r'^\s*\d+\.', na=False)

num_list_mismatch = df[df['starts_with_num_src'] & ~df['starts_with_num_tr']]
len(num_list_mismatch)

0

In [24]:
## uncomment to show
# num_list_mismatch[['element_number', 'text', 'primary_translation']].head(20)

In [25]:
# summarize results
summary = {
    "total_elements": len(df),
    "missing_or_blank_translation": int(mask_empty.sum()),
    "with_primary_error": int(df['primary_error'].notna().sum()),
    "suspect_short_len": int(len(suspect_short)),
    "suspect_long_len": int(len(suspect_long)),
    "newline_mismatch": int(len(suspect_newlines)),
    "star_mismatch": int(len(star_mismatch)),
    "url_lost": int(len(url_lost)),
}
summary

{'total_elements': 29,
 'missing_or_blank_translation': 0,
 'with_primary_error': 0,
 'suspect_short_len': 21,
 'suspect_long_len': 0,
 'newline_mismatch': 0,
 'star_mismatch': 0,
 'url_lost': 0}

### Langid classification
This section applies language-identification labels across the translated elements to get a broader view of how the `primary_translation` output is being classified and to help spot any items that may still contain substantial source-language content.

In [26]:
# langid detect language random sample (important to realize that langid is leaky)

import langid

# Sample a subset to keep it quick
sample = df.sample(min(300, len(df)), random_state=42).copy()

def detect_lang_safe(text):
    if not isinstance(text, str) or not text.strip():
        return "unknown"
    return langid.classify(text)[0]

sample['detected_lang'] = sample['primary_translation'].apply(detect_lang_safe)
sample['detected_lang'].value_counts()

detected_lang
zh    29
Name: count, dtype: int64

In [27]:
# inspect all elements

import langid
import pandas as pd

df = pd.DataFrame(elements)

def detect_lang_safe(text):
    if not isinstance(text, str) or not text.strip():
        return "unknown"
    return langid.classify(text)[0]

# Run on the full set
df['detected_lang'] = df['primary_translation'].apply(detect_lang_safe)

df['detected_lang'].value_counts()

detected_lang
zh    29
Name: count, dtype: int64

In [28]:
df_problem = df[df['detected_lang'].isin(['en','zh','fr','ka','he','lv'])]
df_problem[['element_number','word_style','text','primary_translation']]

,element_number,word_style,text,primary_translation
0,1,h1,Introduction,簡介
1,2,h2,Welcome,歡迎
2,3,body,This course is designed for the person that wa...,本課程專為想要了解成為或作為耶穌跟隨者有何意義的人而設計。許多尋求這方面知識的人明白，他們必...
3,4,body,This is a **self-directed study** to assist yo...,這是一份**自主學習**材料，旨在協助你尋找答案。它的目的是帶領你快速瀏覽精選的聖經書卷，為...
4,5,body,The course consists of **two parts**: the firs...,本課程包含**兩個部分**：第一部分涵蓋基礎知識，可在兩週內完成。第二部分「深入探討」聖經，...
5,6,body,"After completing this course, you will be **eq...",完成本課程後，你將**得到裝備、獲得能力並充滿熱忱地**繼續學習聖經。
6,7,h2,The Non-Sequential Reading Approach,非順序讀經法
7,8,body,Most people that are new to the Bible approach...,大多數剛接觸聖經的人，會像閱讀其他書籍一樣來讀聖經——他們從頭開始，按順序一直讀到最後。
8,9,body,"However, this often leads to frustration becau...",然而，這往往會帶來挫折感，因為線性的閱讀方式無法幫助你輕易地***在閱讀的當下***明白聖經...
9,10,h2,Getting Started,準備開始


In [29]:
# Check elements flagged as English

import langid

suspected_english = []

for el in elements:
    tr = el['primary_translation']
    lang, conf = langid.classify(tr)
    if lang == 'en' and len(tr) > 20:  # avoid tiny text
        suspected_english.append((el['element_number'], tr, conf))

len(suspected_english)

0

In [30]:
suspected_english

[]

In [31]:
# inspect an element by index number
elements[9]

{'element_number': 10,
 'element_type': 'paragraph',
 'word_style': 'h2',
 'text': 'Getting Started',
 'element_id': 'f1e071f3c3e5',
 'tokens': 2,
 'batch_number': 4,
 'primary_translation': '準備開始',
 'primary_translation_model': 'gemini-3.1-pro-preview',
 'primary_error': None,
 'evaluator_ran': None,
 'evaluator_passed': None,
 'evaluator_feedback': None,
 'evaluator_error': None,
 'fallback_translation': None,
 'fallback_translation_model': None,
 'fallback_error': None,
 'final': None,
 'final_model': None}

## Create a Word document based on primary translation only
- Save back to Word using primary translation only (no evaluation and finalization stages)
- Uses a standalone definition function rather than calling the function from workflow_helpers.py
- Saves a new Word document with the original English first and the translated language appended to the end of the document (non-interlinear)

In [ ]:
# Read original word filename and target language from metadata header

docxfilename = metadata.get("docxfilename")
target_language = metadata.get("target_language")

missing = [
    name for name, value in {
        "docxfilename": docxfilename,
        "target_language": target_language,
    }.items()
    if not value
]

if missing:
    raise ValueError(
        f"Missing required metadata field(s) in checkpoint: {', '.join(missing)}"
    )

print("Word source filename:", docxfilename)
print("Target language:", target_language)

Word source filename: custom_word_styles_example.docx
Target language: Traditional Chinese


### Generate the primary-translation Word output

- **Important:** Confirm that the loaded checkpoint metadata points to the correct source DOCX file and target language.
- This step exports the `primary_translation` field, since this notebook is still working at the primary-translation stage.
- The export uses the original Word file as the template/base document and appends the translated content to the end.

In [34]:
from pathlib import Path
from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

word_dir = Path("word_files")

translated_language = target_language
template_path = word_dir / docxfilename

if not template_path.exists():
    raise FileNotFoundError(f"Template file not found: {template_path}")

out_path = word_dir / workflow_helpers.build_output_path_from_base(
    docxfilename=docxfilename,
    translated_language=translated_language,
    text_field="primary_translation",
)

print("Template DOCX:", template_path)
print("Saving output to:", out_path)

workflow_helpers.export_appended_translation_to_docx(
    elements=elements,
    template_path=template_path,
    out_path=out_path,
    text_field="primary_translation",
    page_break_before_translation=False,
)

print("Done.")

Template DOCX: word_files\custom_word_styles_example.docx
Saving output to: word_files\custom_word_styles_example_Traditional_Chinese_primary_translation_20260411_154835.docx
Done.


## Notebook checkpoint and handoff

This notebook inspected the primary translation checkpoint by performing the following steps:

- loaded a metadata-aware checkpoint from the prior stage
- reviewed checkpoint metadata and primary translation state
- checked for unchanged source/translation pairs
- checked for likely English leakage in the translated output
- checked for missing or blank `primary_translation` values
- reviewed simple source/translation length-ratio signals for suspicious truncation or expansion
- generated a Word document using the `primary_translation` field for visual review

### Output of this notebook
The main outputs of this notebook are:

- inspection results shown in the notebook
- a Word document generated from the `primary_translation` field for review purposes

This notebook may also help identify whether the primary translation is ready to proceed to evaluation.

### Scope of this notebook
This notebook focuses on review and inspection of the primary translation stage only.

At this stage:

- `primary_translation` is treated as the working translated text
- evaluator fields remain unprocessed or only partially relevant depending on the checkpoint loaded
- no evaluator decisions are made here
- no fallback translation is run here
- no final output artifacts are produced here

### Next step
If the primary translation looks satisfactory, proceed to the evaluation stage.

Go to: `4_evaluate_primary_translation.ipynb`